# Imaged FOVs — live acquisition progress map

Watches one round of a running acquisition and draws a square over each FOV
position as its image file actually appears on disk — a live version of
`before_imaging/<variant>/02_create_positions_from_boundaries.ipynb`'s own
FOV-layout plot, so you can see *where* on the coverslip the microscope
currently is without switching to Steve.

Background: the real Steve low-mag mosaic photo when the boundaries were
derived from one (`02_create_boundary_from_mosaic.ipynb`, needs
`data/mosaic10x/*.msc` on disk), otherwise the same schematic tissue-boundary
outline `02_create_positions_from_boundaries.ipynb` draws.

**FOV coloring**: gray outline = planned, not yet imaged; filled = imaged.
A FOV is additionally flagged (a different fill color) if HAL's own `.off`
focus-lock sidecar shows `good-offset` was 1 in **fewer than
`MIN_GOOD_OFFSET_FRAMES`** frames of that FOV's stack. The default
(`MIN_GOOD_OFFSET_FRAMES = 1`) flags only a FOV where the two-spot focus lock
was never found at all (`good-offset` is 0 for *every* frame) -- not the
normal case of losing lock only once the z-sweep goes deeper than the lock's
tracking range (which flips `good-offset` from 1→0 partway through most
FOVs' stacks). Raise `MIN_GOOD_OFFSET_FRAMES` to also flag FOVs where focus
was found only briefly (a weak/marginal lock), at the cost of also flagging
some FOVs going through that normal partial transition. This is a
heuristic, not the same signal as Dave's own "Check Focus Lock" warning
(which only exists, unreliably, in storm_control's generic rotating debug
log — see `prompt_history/` for why that path wasn't used here).

**Usage**: run every cell once, then run the last cell and leave it running
while HAL/Dave images the round you're watching. Interrupt the kernel
(Jupyter's stop button) to end the live loop cleanly at any time — the last
drawn state is kept and saved to `figures/`.

## 1 — Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, clear_output

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs    import get_fov_geometry
from MERci.acquisition.positions  import (
    resolve_boundaries_source_dir, discover_boundary_files,
    load_boundary_polygon, load_hole_polygons,
)
from MERci.acquisition.mosaic     import load_steve_mosaic, assemble_mosaic_canvas, MosaicCanvas
from MERci.analysis.stage_z       import focus_lock_summary_for_fov

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # must match what HAL is writing
NOTEBOOK_NAME = "imaged_fovs"   # used to namespace this notebook's cache + figure files

MICROSCOPE = "MF3"   # MF2-MF5: 0.108 um/px, 2048 px | MFX/ST2: 0.0878 um/px, 2304 px
OBJECTIVE  = "60X"   # ST2 only, for now: "60X" (0.0878 um/px) or "40X" (0.1317 um/px, scaled placeholder)
pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE, OBJECTIVE)
fov_size_um = pixel_size_um * image_size_px

# Which round to watch. None = auto-detect (prefer a round with SOME but not
# ALL FOVs already imaged -- i.e. one HAL/Dave is actively writing right now;
# fall back to the most recently active round, then the first round, if none
# is in progress yet). Set an explicit imaging_round number to override.
ROUND_ID = None

POLL_INTERVAL_SEC = 5     # must be well under the time to acquire one FOV
MAX_RUNTIME_MIN    = 240  # safety cap -- stop polling (not an error) if a round
                           # genuinely never finishes in this many minutes

UNIMAGED_EDGECOLOR = "0.55"
IMAGED_COLOR       = "gold"
WARNING_COLOR      = "red"   # focus-lock-flag color -- see markdown above

# Flag a FOV if good-offset == 1 in FEWER than this many frames of its stack.
# 1 (default) reproduces the original "never locked at all" behavior (flag
# only if good-offset is 0 for every frame). Raise it to also flag FOVs where
# focus was found only briefly -- e.g. MIN_GOOD_OFFSET_FRAMES=5 flags any FOV
# with fewer than 5 good-offset=1 frames in its whole stack.
MIN_GOOD_OFFSET_FRAMES = 1

# Mosaic canvas assembly parameters -- must match whatever
# 02_create_boundary_from_mosaic.ipynb was actually run with, or the
# recreated image won't line up with the boundary/FOV positions.
MOSAIC_KEEP_OBJECTIVES  = ["10x"]
MOSAIC_WORKING_PIXEL_UM = 5.0

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
# Deliberately SAMPLE_DIR/figures/, not analysis/figures/ like other notebooks
# (NOTEBOOK_GUIDELINES.md #6) -- this notebook's output is a live progress
# view meant to be checked at a glance during acquisition, not filed away
# under analysis/ alongside post-hoc QC figures.
FIGURES_DIR = SAMPLE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name  : {SAMPLE_NAME}")
print(f"FOVs (total) : {meta.n_fovs}")
print(f"Rounds       : {sorted(meta.rounds)}")
print(f"FOV size     : {fov_size_um:.1f} um  ({MICROSCOPE}, {OBJECTIVE})")

## 3 — Pick the round to watch

Scans every round's expected files once (cheap `.exists()`/`.stat()` calls,
no image reads) to auto-detect which round is currently being written, if
`ROUND_ID` was left as `None`. If nothing is actively in progress (e.g. this
is run during the fluidics gap between one round finishing and the next
starting -- fluidics runs strictly between rounds' imaging loops, never
mid-round, so the "next round" always has zero files at that point and would
otherwise be invisible to this scan), it points at the round *after* the most
recently completed one instead of the completed one itself, so the notebook
is ready and waiting rather than showing an already-finished round as "done."


In [ ]:
def _round_progress(round_id):
    # (n_imaged, latest_mtime_or_None) for round_id, over every planned FOV.
    series = meta.series_for_round(round_id)
    n_imaged, latest = 0, None
    for fov_id in sorted(meta.fovs):
        paths = [s.resolve_path(fov_id, config.image_suffix) for s in series]
        existing = [p for p in paths if p.exists()]
        if existing:
            n_imaged += 1
            mtime = max(_path_mtime(p) for p in existing)
            latest = mtime if latest is None else max(latest, mtime)
    return n_imaged, latest


def _path_mtime(path):
    # A .zarr store is a directory -- its own mtime doesn't reliably reflect
    # a chunk file written inside it, so fall back to the newest member file.
    path = Path(path)
    if path.is_dir():
        member_mtimes = [f.stat().st_mtime for f in path.rglob("*") if f.is_file()]
        return max(member_mtimes) if member_mtimes else path.stat().st_mtime
    return path.stat().st_mtime


def detect_active_round():
    best_round, best_latest, best_in_progress = None, -1.0, False
    for round_id in sorted(meta.rounds):
        n_imaged, latest = _round_progress(round_id)
        if latest is None:
            continue
        in_progress = n_imaged < meta.n_fovs
        if (in_progress, latest) > (best_in_progress, best_latest):
            best_round, best_latest, best_in_progress = round_id, latest, in_progress

    if best_round is None:
        return sorted(meta.rounds)[0]   # nothing imaged anywhere yet -- start of the sequence

    if not best_in_progress:
        # Nothing is actively in progress -- best_round is just the most
        # recently completed one. Point at the round right after it instead
        # (e.g. during the fluidics gap before it starts, which otherwise
        # has zero files and is invisible to this scan), unless best_round
        # is already the last round in the experiment.
        round_ids = sorted(meta.rounds)
        idx = round_ids.index(best_round)
        if idx + 1 < len(round_ids):
            return round_ids[idx + 1]

    return best_round


if ROUND_ID is None:
    ROUND_ID = detect_active_round()
    print(f"Auto-detected ROUND_ID = {ROUND_ID} (override the parameter above to pick a different round)")
else:
    print(f"Watching explicit ROUND_ID = {ROUND_ID}")

ROUND_SERIES = meta.series_for_round(ROUND_ID)
if not ROUND_SERIES:
    raise ValueError(f"round_info.csv has no series for imaging_round={ROUND_ID}")

n_imaged_now, _ = _round_progress(ROUND_ID)
print(f"Round {ROUND_ID}: {n_imaged_now}/{meta.n_fovs} FOV(s) already imaged.")

## 4 — Load the background (real mosaic photo, else schematic outline)

Reuses `02_create_positions_from_boundaries.ipynb`'s own boundary-resolution
and mosaic-assembly code so this always agrees with whatever that notebook
actually used, and caches the assembled canvas the same way (skipped if
`BOUNDARY_SOURCE != "from_mosaic"` or no `.msc` file is present yet).

In [ ]:
POSITIONS_DIR = SAMPLE_DIR / "positions"
BOUNDARY_DIR, BOUNDARY_SOURCE = resolve_boundaries_source_dir(POSITIONS_DIR, None)

boundaries, MODE = ([], None)
boundary_polys   = []
holes            = []
mosaic_canvas    = None

if BOUNDARY_DIR is not None and BOUNDARY_DIR.exists():
    boundaries, MODE = discover_boundary_files(BOUNDARY_DIR)
    boundary_polys    = [load_boundary_polygon(b.path) for b in boundaries]
    holes             = load_hole_polygons(BOUNDARY_DIR)

MOSAIC_DIR = SAMPLE_DIR / "data" / "mosaic10x"
msc_candidates = sorted(MOSAIC_DIR.glob("*.msc")) if BOUNDARY_SOURCE == "from_mosaic" else []

if msc_candidates:
    import json

    msc_path = msc_candidates[0]
    cache_npz    = CACHE_DIR / "mosaic_canvas.npz"
    cache_params = CACHE_DIR / "mosaic_canvas_params.json"
    params_now = {
        "msc_path": str(msc_path), "msc_mtime": msc_path.stat().st_mtime,
        "keep_objectives": MOSAIC_KEEP_OBJECTIVES, "working_pixel_um": MOSAIC_WORKING_PIXEL_UM,
    }

    cached_ok = False
    if cache_npz.exists() and cache_params.exists():
        with open(cache_params) as fh:
            cached_ok = json.load(fh) == params_now

    if cached_ok:
        npz = np.load(cache_npz)
        mosaic_canvas = MosaicCanvas(
            image=npz["image"], covered=npz["covered"],
            origin_um=tuple(npz["origin_um"]), pixel_size_um=float(npz["pixel_size_um"]),
        )
        print(f"Loaded cached mosaic canvas: {cache_npz}")
    else:
        tiles_all = load_steve_mosaic(msc_path)
        tiles = [t for t in tiles_all if t.objective_name in MOSAIC_KEEP_OBJECTIVES]
        mosaic_canvas = assemble_mosaic_canvas(tiles, working_pixel_um=MOSAIC_WORKING_PIXEL_UM)
        np.savez_compressed(
            cache_npz, image=mosaic_canvas.image, covered=mosaic_canvas.covered,
            origin_um=np.array(mosaic_canvas.origin_um), pixel_size_um=mosaic_canvas.pixel_size_um,
        )
        with open(cache_params, "w") as fh:
            json.dump(params_now, fh)
        print(f"Assembled + cached mosaic canvas from {len(tiles)} tile(s): {cache_npz}")

    print(f"Background  : real mosaic photo ({mosaic_canvas.image.shape[1]}x{mosaic_canvas.image.shape[0]} px "
          f"at {mosaic_canvas.pixel_size_um:.2f} um/px)")
else:
    print("Background  : schematic tissue-boundary outline "
          "(no from_mosaic boundaries / .msc file found)")

## 5 — Build the static base plot (one square per planned FOV)

In [ ]:
def _um_to_px(x, y, canvas):
    return ((np.asarray(x) - canvas.origin_um[0]) / canvas.pixel_size_um,
             (np.asarray(y) - canvas.origin_um[1]) / canvas.pixel_size_um)


fov_ids = sorted(meta.fovs)
patch_of = {}

if mosaic_canvas is not None:
    half_px = (fov_size_um / 2) / mosaic_canvas.pixel_size_um

    xs_um = np.array([meta.fovs[f].position[0] for f in fov_ids])
    ys_um = np.array([meta.fovs[f].position[1] for f in fov_ids])
    px, py = _um_to_px(xs_um, ys_um, mosaic_canvas)
    margin_px = 5 * (fov_size_um / mosaic_canvas.pixel_size_um)
    x_min, x_max = px.min() - margin_px, px.max() + margin_px
    y_min, y_max = py.min() - margin_px, py.max() + margin_px

    step_px      = fov_size_um / mosaic_canvas.pixel_size_um
    inch_per_fov = 0.35
    fig_w = float(np.clip((x_max - x_min) / step_px * inch_per_fov, 8, 60))
    fig_h = float(np.clip((y_max - y_min) / step_px * inch_per_fov, 8, 60))

    covered_vals = mosaic_canvas.image[mosaic_canvas.covered]
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.imshow(mosaic_canvas.image, cmap="gray",
              vmin=np.percentile(covered_vals, 1), vmax=np.percentile(covered_vals, 99))

    for poly in boundary_polys:
        bx, by = _um_to_px(*poly.exterior.xy, mosaic_canvas)
        ax.plot(bx, by, "-", lw=1.0, color="tab:blue")
    for hole in holes:
        hx, hy = _um_to_px(*hole.exterior.xy, mosaic_canvas)
        ax.plot(hx, hy, "r--", lw=0.8)

    for fov_id, cx, cy in zip(fov_ids, px, py):
        patch_of[fov_id] = mpatches.Rectangle(
            (cx - half_px, cy - half_px), 2 * half_px, 2 * half_px,
            lw=0.4, edgecolor=UNIMAGED_EDGECOLOR, facecolor="none",
        )
        ax.add_patch(patch_of[fov_id])

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_max, y_min)   # imshow's default origin puts row 0 at the top
    ax.set_xlabel("canvas column (px)")
    ax.set_ylabel("canvas row (px)")
else:
    half = fov_size_um / 2
    fig, ax = plt.subplots(figsize=(8, 7))

    for poly in boundary_polys:
        ax.plot(*poly.exterior.xy, "-", lw=1.0, color="tab:blue")
    for hole in holes:
        ax.fill(*hole.exterior.xy, color="0.85", alpha=0.6)
        ax.plot(*hole.exterior.xy, "k--", lw=0.8)

    for fov_id in fov_ids:
        x, y = meta.fovs[fov_id].position
        patch_of[fov_id] = mpatches.Rectangle(
            (x - half, y - half), fov_size_um, fov_size_um,
            lw=0.4, edgecolor=UNIMAGED_EDGECOLOR, facecolor="none",
        )
        ax.add_patch(patch_of[fov_id])

    ax.invert_yaxis()
    ax.axis("equal")
    ax.set_xlabel("Stage X (um)")
    ax.set_ylabel("Stage Y (um)")

ax.set_title(f"Imaged FOVs — {SAMPLE_NAME}, round {ROUND_ID}")
fig.tight_layout()
display(fig)

## 6 — Live loop: color a square in as its FOV is imaged

Interrupt the kernel to stop — the figure keeps its last drawn state and is
saved to `figures/` either way.

In [ ]:
imaged      = set()
warned      = set()
off_checked = set()   # fov_ids whose .off file has already given a definitive verdict

start_time = time.time()
poll_count = 0

try:
    while len(imaged) < len(fov_ids):
        if (time.time() - start_time) > MAX_RUNTIME_MIN * 60:
            print(f"Stopping: MAX_RUNTIME_MIN={MAX_RUNTIME_MIN} exceeded with "
                  f"{len(imaged)}/{len(fov_ids)} FOVs imaged.")
            break

        poll_count += 1
        newly_imaged = []
        for fov_id in fov_ids:
            if fov_id in imaged:
                continue
            paths = [s.resolve_path(fov_id, config.image_suffix) for s in ROUND_SERIES]
            existing = [p for p in paths if p.exists()]
            if existing:
                imaged.add(fov_id)
                newly_imaged.append((fov_id, existing[0]))

        for fov_id, image_path in newly_imaged:
            patch_of[fov_id].set_facecolor(IMAGED_COLOR)
            patch_of[fov_id].set_edgecolor(IMAGED_COLOR)
            patch_of[fov_id].set_alpha(0.6)

        # (Re-)check .off files for every imaged-but-not-yet-classified FOV --
        # the .off sidecar can lag behind the main image file, so a FOV that
        # was too new to classify last cycle is retried here, not skipped.
        pending = [f for f in imaged if f not in off_checked]
        for fov_id in pending:
            paths = [s.resolve_path(fov_id, config.image_suffix) for s in ROUND_SERIES]
            existing = [p for p in paths if p.exists()]
            summary = focus_lock_summary_for_fov(existing[0]) if existing else None
            if summary is None:
                continue   # .off not written/flushed yet -- retry next poll
            off_checked.add(fov_id)
            n_good_frames = summary["n_frames"] - summary["n_bad_frames"]
            flagged = n_good_frames < MIN_GOOD_OFFSET_FRAMES
            if flagged:
                warned.add(fov_id)
                patch_of[fov_id].set_facecolor(WARNING_COLOR)
                patch_of[fov_id].set_edgecolor(WARNING_COLOR)
                patch_of[fov_id].set_alpha(0.8)

        elapsed = time.time() - start_time
        rate    = len(imaged) / elapsed if elapsed > 0 else 0.0
        eta_sec = (len(fov_ids) - len(imaged)) / rate if rate > 0 else float("nan")
        eta_str = f"{eta_sec / 60:.1f} min" if rate > 0 else "n/a"

        clear_output(wait=True)
        display(fig)
        print(f"poll #{poll_count} | elapsed {elapsed / 60:.1f} min | "
              f"imaged {len(imaged)}/{len(fov_ids)} ({100 * len(imaged) / len(fov_ids):.1f}%) | "
              f"warnings: {len(warned)} | ETA {eta_str} | next check in {POLL_INTERVAL_SEC}s")

        if len(imaged) < len(fov_ids):
            time.sleep(POLL_INTERVAL_SEC)
except KeyboardInterrupt:
    print(f"Stopped by user at {len(imaged)}/{len(fov_ids)} FOVs imaged.")

fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.progress.png", dpi=150)
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.progress.png'}")
